In [ ]:
import sys
import os

print("====== STEP 1: CLONING BENCHMARK AND ANOMALYCLIP ======")
# 1. Define public repository details
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
repo_url = f"https://github.com/{BENCHMARK_REPOSITORY}.git"
target_dir = "/kaggle/working/Natural-Corruption-Robustness"
anomalyclip_dir = "/kaggle/working/AnomalyCLIP"

# 3. Clone/update this benchmark repository
if not os.path.exists(target_dir):
    print(f"Cloning public benchmark repository into {target_dir}...")
    clone_status = os.system(f"git clone {repo_url} {target_dir}")
    if clone_status == 0:
        print("Repository cloned successfully.")
    else:
        raise RuntimeError("Git clone failed. Check your token permissions.")
else:
    print("Repository directory already exists. Pulling latest changes...")
    pull_status = os.system(f"git -C {target_dir} pull --ff-only")
    if pull_status == 0:
        print("Repository updated successfully.")
    else:
        print("Repository update skipped or failed. Continuing with existing checkout.")

# 4. Clone/update the official AnomalyCLIP implementation used by the wrapper
if not os.path.exists(anomalyclip_dir):
    print(f"Cloning official AnomalyCLIP into {anomalyclip_dir}...")
    clone_status = os.system(f"git clone https://github.com/zqhang/AnomalyCLIP.git {anomalyclip_dir}")
    if clone_status == 0:
        print("AnomalyCLIP cloned successfully.")
    else:
        raise RuntimeError("Failed to clone https://github.com/zqhang/AnomalyCLIP")
else:
    print("AnomalyCLIP directory already exists. Pulling latest changes...")
    pull_status = os.system(f"git -C {anomalyclip_dir} pull --ff-only")
    if pull_status == 0:
        print("AnomalyCLIP updated successfully.")
    else:
        print("AnomalyCLIP update skipped or failed. Continuing with existing checkout.")

# 5. Register path environment for Python
if anomalyclip_dir not in sys.path:
    sys.path.insert(0, anomalyclip_dir)
    print("Registered AnomalyCLIP path to sys.path.")

print("\n====== STEP 2: INSTALLING SYSTEM DEPENDENCIES ======")
os.system("pip install -q open-clip-torch scipy opencv-python scikit-learn scikit-image ftfy regex tqdm tabulate timm==0.6.12 torchsummary seaborn dash-table thop kornia==0.6.9 ipdb tiktoken einops")
print("Core dependencies installed successfully.")
print("\nENVIRONMENT READY.")


In [ ]:
import sys
import os
import gc
from pathlib import Path
import torch

# 1. Set the required environment variable for AnomalyCLIP.
os.environ["ANOMALYCLIP_ROOT"] = "/kaggle/working/AnomalyCLIP"

# 2. Append the zero-shot framework harness package directory into Python's path search
harness_path = '/kaggle/working/Natural-Corruption-Robustness'
if harness_path not in sys.path:
    sys.path.insert(0, harness_path)

# Ensure the cloned AnomalyCLIP path is also visible to Python.
anomalyclip_path = '/kaggle/working/AnomalyCLIP'
if anomalyclip_path not in sys.path:
    sys.path.insert(0, anomalyclip_path)

from harness.runner import run_evaluation

# ==============================================================================
# DATASET & MODEL CONFIGURATION
# ==============================================================================
# Choose exactly one dataset for this notebook session: "mvtec" or "visa".
# DATASET_NAME = "mvtec"
DATASET_NAME = "visa"

MODEL_NAME = "AnomalyCLIP"
# False: evaluate every concrete corruption independently. True: evaluate the
# balanced categories below, assigning each image one operation in that group.
USE_CATEGORIZED_CORRUPTIONS = False
# Must match base_seed in demo_categorized.ipynb and its CSV generator.
CATEGORIZED_CORRUPTION_SEED = 123

UNCATEGORIZED_CORRUPTION_TYPES = [
    "gaussian_noise",
    "shot_noise",
    "impulse_noise",
    "defocus_blur",
    "motion_blur",
    "zoom_blur",
    "brightness",
    "contrast",
]
CATEGORIZED_CORRUPTION_TYPES = ["noise", "blur", "photometric", "geometric"]
CORRUPTION_TYPES = (
    CATEGORIZED_CORRUPTION_TYPES
    if USE_CATEGORIZED_CORRUPTIONS else UNCATEGORIZED_CORRUPTION_TYPES
)
SEVERITY_LEVELS = [1, 2, 3]
BATCH_SIZE = 2  # Increase to 8 or 16 on larger Kaggle GPUs if memory allows.
CORRUPTION_CACHE_FORMAT = "png"  # Use "jpeg" if you want ImageNet-C-style cached files.

# ==============================================================================
# ENVIRONMENT PATH CONFIGURATION
# ==============================================================================
MVTEC_PATH = "/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection"
VISA_PATH = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
CORRUPTION_CACHE_ROOT = "/kaggle/working/corruption_cache"
ANOMALYCLIP_ROOT = "/kaggle/working/AnomalyCLIP"

DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa"}:
    raise ValueError("DATASET_NAME must be either 'mvtec' or 'visa'.")

IS_MVTEC = DATASET_NAME == "mvtec"
selected_dataset = "MVTec AD" if IS_MVTEC else "VisA"
PERSISTENT_CORRUPTION_PLAN = str(
    Path(harness_path) / (
        "mvtec_persistent_corruptions.csv"
        if IS_MVTEC else "persistent_corruptions.csv"
    )
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# AnomalyCLIP needs a prompt learner checkpoint. The official default is
# checkpoints/9_12_4_multiscale/epoch_15.pth; Kaggle inputs are also scanned.
def _valid_path(value):
    if not value:
        return None
    path = Path(value)
    return path if str(path) != "." and path.exists() else None


def _kaggle_input_roots():
    input_root = Path("/kaggle/input")
    return sorted(input_root.iterdir()) if input_root.exists() else []


checkpoint_candidates = [
    _valid_path(os.environ.get("ANOMALYCLIP_CHECKPOINT")),
    Path(ANOMALYCLIP_ROOT) / "checkpoints/9_12_4_multiscale/epoch_15.pth",
    Path(ANOMALYCLIP_ROOT) / "checkpoints/9_12_4_multiscale_visa/epoch_15.pth",
    Path("/kaggle/input/anomalyclip-checkpoints/epoch_15.pth"),
    Path("/kaggle/input/anomalyclip-checkpoints"),
    *_kaggle_input_roots(),
]
checkpoint_candidates = [path for path in checkpoint_candidates if path is not None]
ANOMALYCLIP_CHECKPOINT = None
for path in checkpoint_candidates:
    if path.is_file() and path.suffix == ".pth":
        ANOMALYCLIP_CHECKPOINT = str(path)
        break
    if path.is_dir():
        default = path / "epoch_15.pth"
        if default.exists():
            ANOMALYCLIP_CHECKPOINT = str(default)
            break
        pth_files = sorted(path.rglob("*.pth"))
        if pth_files:
            ANOMALYCLIP_CHECKPOINT = str(pth_files[-1])
            break

if ANOMALYCLIP_CHECKPOINT is None:
    expected = "\n".join(f"  - {path}" for path in checkpoint_candidates if str(path))
    raise FileNotFoundError(
        "AnomalyCLIP checkpoint not found. Add a Kaggle input containing "
        "epoch_15.pth or set ANOMALYCLIP_CHECKPOINT. Checked:\n"
        f"{expected}"
    )

model_kwargs = {
    "AnomalyCLIP": {
        "anomalyclip_root": ANOMALYCLIP_ROOT,
        "checkpoint_path": ANOMALYCLIP_CHECKPOINT,
        "image_size": 518,
        "features_list": [6, 12, 18, 24],
        "feature_map_layer": [0, 1, 2, 3],
        "depth": 9,
        "n_ctx": 12,
        "t_n_ctx": 4,
        "dpam_layer": 20,
    }
}

print("LAUNCHING ISOLATED BENCHMARK")
print(f"Model:      {MODEL_NAME}")
print(f"Dataset:    {selected_dataset}")
print(f"Corruption: {CORRUPTION_TYPES} @ severities {SEVERITY_LEVELS}")
print(f"Categorized protocol: {USE_CATEGORIZED_CORRUPTIONS}")
if USE_CATEGORIZED_CORRUPTIONS:
    print(f"Persistent plan: {PERSISTENT_CORRUPTION_PLAN}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device:     {DEVICE}")
print(f"Checkpoint: {ANOMALYCLIP_CHECKPOINT}")
print(f"Outputs:    {OUTPUT_ROOT}")
print(f"Cache:      {CORRUPTION_CACHE_ROOT} ({CORRUPTION_CACHE_FORMAT})\n")

# ==============================================================================
# CONDITIONAL EVALUATION EXECUTION
# ==============================================================================
if IS_MVTEC:
    print(f"--- Starting isolated MVTec AD run for {MODEL_NAME} ---")
    try:
        run_evaluation(
            mvtec_root=MVTEC_PATH,
            visa_root=None,
            output_root=OUTPUT_ROOT,
            models=[MODEL_NAME],
            model_kwargs=model_kwargs,
            device=DEVICE,
            dataset="mvtec",
            corruption_types=CORRUPTION_TYPES,
            severity_levels=SEVERITY_LEVELS,
            batch_size=BATCH_SIZE,
            corruption_cache_root=CORRUPTION_CACHE_ROOT,
            corruption_cache_format=CORRUPTION_CACHE_FORMAT,
            categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
            categorized_corruption_plans={DATASET_NAME: PERSISTENT_CORRUPTION_PLAN},
            corruption_seed=(CATEGORIZED_CORRUPTION_SEED if USE_CATEGORIZED_CORRUPTIONS else None),
        )
        print(f"Successful MVTec evaluation completed for {MODEL_NAME}.")
    except Exception as e:
        print(f"ERROR during MVTec run for {MODEL_NAME}: {str(e)}")
        raise

else:
    print(f"--- Starting isolated VisA run for {MODEL_NAME} ---")
    try:
        run_evaluation(
            mvtec_root=None,
            visa_root=VISA_PATH,
            output_root=OUTPUT_ROOT,
            models=[MODEL_NAME],
            model_kwargs=model_kwargs,
            device=DEVICE,
            dataset="visa",
            corruption_types=CORRUPTION_TYPES,
            severity_levels=SEVERITY_LEVELS,
            batch_size=BATCH_SIZE,
            corruption_cache_root=CORRUPTION_CACHE_ROOT,
            corruption_cache_format=CORRUPTION_CACHE_FORMAT,
            categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
            categorized_corruption_plans={DATASET_NAME: PERSISTENT_CORRUPTION_PLAN},
            corruption_seed=(CATEGORIZED_CORRUPTION_SEED if USE_CATEGORIZED_CORRUPTIONS else None),
        )
        print(f"Successful VisA evaluation completed for {MODEL_NAME}.")
    except Exception as e:
        print(f"ERROR during VisA run for {MODEL_NAME}: {str(e)}")
        raise

# ==============================================================================
# POST-RUN CLEANUP
# ==============================================================================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "="*70)
print(f"EXECUTION FINISHED FOR {MODEL_NAME} ON {selected_dataset}")
print(f"Check your '{OUTPUT_ROOT}' directory to collect the generated CSV files.")
print("="*70)
